# Requirements

In [12]:
# Libraries
import rasterio
import geopandas as gpd
from tqdm import tqdm
from pathlib import Path

from beak.utilities.raster_processing import calculate_distance_from_raster, calculate_distance_from_raster_gdal_base_fn
from beak.utilities.io import data_folder, save_raster, check_path, create_file_list
from beak.utilities.conversions import create_binary_raster

# Set base data folder path
data_folder = data_folder()
vector_file_name = "SGMC_Geology_carbonate"

# Paths
cma_name = "regional_tusk_great_basin_102008_500"
base_raster_file = data_folder / "processed" / cma_name / "base_raster" / "template_raster.tif"
vector_files = data_folder / "raw" / "remote_sensing" / "regional_scale" / "tusk_great_basin" / "alteration"

out_path = data_folder / "processed" / cma_name / "unified" / "remote_sensing" / "alteration"
check_path(out_path)


WindowsPath('S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Bearbeitung/GitHub/beak-ta3/src/beak/data/processed/regional_tusk_great_basin_102008_500/unified/remote_sensing/distance/alteration')

In [13]:
files = create_file_list(vector_files, extensions=[".shp"], recursive=False)

for file in tqdm(files):
    base_raster = rasterio.open(base_raster_file)
    
    gdf = gpd.read_file(file)
    gdf = gdf.to_crs(base_raster.crs)
    
    binary_raster, meta = create_binary_raster(
        geodataframe=gdf,
        base_raster=base_raster,
        query=None,
        return_meta=True,
    )
    
    # Result
    out_file = out_path / f"{file.stem}_proximity.tif"
    distance_array, distance_meta = calculate_distance_from_raster(input_data=(binary_raster, meta), input_mask=base_raster)
    save_raster(out_file, distance_array, metadata=distance_meta, overwrite=True)
    
    # Also save as imputed (for simplicity to call layers later on)
    out_file_imputed = str(out_path).replace("unified", "unified_imputed")
    out_file_imputed = Path(out_file_imputed) / f"{file.stem}_proximity.tif"
    
    check_path(out_file_imputed.parent)
    save_raster(out_file_imputed, distance_array, metadata=distance_meta, overwrite=True)
    

100%|██████████| 2/2 [01:35<00:00, 47.96s/it]
